In [11]:
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
SUBSET = 'others'  # "eu" or "others"
RESULTS_DIR = '../experiments_ancestry'
COHEN_ID = 100

metadata = pd.read_csv('clean_cohen_table_with_ethnicity_cat_sub_pmid.tsv', delimiter='\t', usecols=['image_id', 'patient_id', 'ethnicity_category', 'ethnicity_subgroup'])
# relevant column: 'ethnicity_category'

In [13]:
def prep_csv(df, is_pickle=False):
    df = df.groupby('img_name').agg(lambda x: list(x)).reset_index()
    if not is_pickle:
        df.representations = df.representations.apply(lambda x: np.array([json.loads(i) for i in x]))
    # df.class_conf = df.class_conf.apply(lambda x: np.array([json.loads(i) for i in x]))
    df.img_name = df.img_name.apply(lambda x: x.split('_')[0])
    return df

class_confs_EU = prep_csv(pd.read_csv(os.path.join(RESULTS_DIR, 'cohen_encodings_EU.csv'), delimiter=';'))
class_confs_others = prep_csv(pd.read_csv(os.path.join(RESULTS_DIR, 'cohen_encodings_Others.csv'), delimiter=';'))

In [14]:
# Get correct class id for Cohen syndrome
cohen_idxs = []
for i in range(1,6):
    lut = json.loads(pd.read_csv(RESULTS_DIR + f'/lookup_table_gmdb_v1.1.0_fold{i}.txt', delimiter=';').values[0][0])
    cohen_idxs.append(np.where(np.array(lut) == COHEN_ID)[0][0])
cohen_idxs = np.array(cohen_idxs)


In [15]:
# Get the correct rank per model, per image, for EU+EU*
correct_ranks_EU = []
for img_idx in range(len(class_confs_EU)):
    ranks_img = []
    for model in range(5):
        ranks = np.argsort(json.loads(class_confs_EU.class_conf.values[img_idx][model]))[::-1]
        ranks_img.append(np.where(ranks == cohen_idxs[model])[0][0])
    correct_ranks_EU.append(ranks_img)

# Get the correct rank per model, per image, for EU+Others
correct_ranks_others = []
for img_idx in range(len(class_confs_EU)):
    ranks_img = []
    for model in range(5):
        ranks = np.argsort(json.loads(class_confs_others.class_conf.values[img_idx][model]))[::-1]
        ranks_img.append(np.where(ranks == cohen_idxs[model])[0][0])
    correct_ranks_others.append(ranks_img)


In [16]:
## Overall performance for EU+EU* and EU+Others
# EU+EU*
print("EU+EU*:")
correct_ranks_EU = np.array(correct_ranks_EU)
print(f"Mean rank: {np.mean(correct_ranks_EU):.3f}")

for n in [1,5,10]:
    accs = np.sum(correct_ranks_EU<n, axis=0)/(len(correct_ranks_EU))
    print(f"Top-{n}: {np.mean(accs)*100:.2f}% +- {np.std(accs)*100:.2f}%")

#EU+Others
print("\nEU+Others:")
correct_ranks_others = np.array(correct_ranks_others)
print(f"Mean rank: {np.mean(correct_ranks_others):.3f}")

for n in [1,5,10]:
    accs = np.sum((correct_ranks_others)<n, axis=0)/(len(correct_ranks_others))
    print(f"Top-{n}: {np.mean(accs)*100:.2f}% +- {np.std(accs)*100:.2f}%")

EU+EU*:
Mean rank: 4.896
Top-1: 64.30% +- 3.50%
Top-5: 86.19% +- 1.58%
Top-10: 90.94% +- 0.79%

EU+Others:
Mean rank: 5.498
Top-1: 69.89% +- 2.90%
Top-5: 87.17% +- 0.86%
Top-10: 91.02% +- 0.77%


In [17]:
## Per ancestry performance (EU+EU*)
# relevant columns: ethnicity_category
all_ancs_accs_eu = []
all_ancs_stdevs_eu = []
rel_ancs = np.unique(metadata.ethnicity_category.values)
for anc in rel_ancs:
    anc_mask = metadata[metadata.ethnicity_category == anc].index.tolist()
    print(f"{anc} (n={len(anc_mask)}):")
    top_n_accs = []
    top_n_stdevs = []
    for n in [1,5,10]:
        accs = np.sum((correct_ranks_EU[anc_mask])<n, axis=0)/(len(correct_ranks_EU[anc_mask]))
        acc = np.mean(accs)
        stdev = np.std(accs)
        print(f"\tTop-{n}: {acc*100:.2f}% +- {stdev*100:.2f}%")
        top_n_accs.append(acc)
        top_n_stdevs.append(stdev)
    all_ancs_accs_eu.append(top_n_accs)
    all_ancs_stdevs_eu.append(top_n_stdevs)

African (n=5):
	Top-1: 72.00% +- 9.80%
	Top-5: 92.00% +- 9.80%
	Top-10: 96.00% +- 8.00%
Asian (n=89):
	Top-1: 65.17% +- 4.49%
	Top-5: 88.31% +- 1.15%
	Top-10: 91.46% +- 0.90%
European (n=156):
	Top-1: 66.92% +- 3.02%
	Top-5: 87.95% +- 2.27%
	Top-10: 92.82% +- 1.18%
Mixed ancestry (n=6):
	Top-1: 63.33% +- 16.33%
	Top-5: 96.67% +- 6.67%
	Top-10: 100.00% +- 0.00%
Others (n=6):
	Top-1: 3.33% +- 6.67%
	Top-5: 20.00% +- 12.47%
	Top-10: 50.00% +- 14.91%
Unknown (n=3):
	Top-1: 13.33% +- 16.33%
	Top-5: 33.33% +- 0.00%
	Top-10: 33.33% +- 0.00%


In [18]:
## Per ancestry performance (EU+Others)
# relevant columns: ethnicity_category
all_ancs_accs_others = []
all_ancs_stdevs_others = []
rel_ancs = np.unique(metadata.ethnicity_category.values)
for anc in rel_ancs:
    anc_mask = metadata[metadata.ethnicity_category == anc].index.tolist()
    print(f"{anc} (n={len(anc_mask)}):")
    top_n_accs = []
    top_n_stdevs = []
    for n in [1,5,10]:
        accs = np.sum((correct_ranks_others[anc_mask])<n, axis=0)/(len(correct_ranks_others[anc_mask]))
        acc = np.mean(accs)
        stdev = np.std(accs)
        print(f"\tTop-{n}: {acc*100:.2f}% +- {stdev*100:.2f}%")
        top_n_accs.append(acc)
        top_n_stdevs.append(stdev)
    all_ancs_accs_others.append(top_n_accs)
    all_ancs_stdevs_others.append(top_n_stdevs)

African (n=5):
	Top-1: 88.00% +- 9.80%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
Asian (n=89):
	Top-1: 73.71% +- 3.30%
	Top-5: 86.74% +- 1.93%
	Top-10: 91.46% +- 1.52%
European (n=156):
	Top-1: 70.13% +- 2.71%
	Top-5: 88.72% +- 1.44%
	Top-10: 92.18% +- 1.37%
Mixed ancestry (n=6):
	Top-1: 73.33% +- 17.00%
	Top-5: 96.67% +- 6.67%
	Top-10: 100.00% +- 0.00%
Others (n=6):
	Top-1: 10.00% +- 13.33%
	Top-5: 60.00% +- 8.16%
	Top-10: 66.67% +- 0.00%
Unknown (n=3):
	Top-1: 26.67% +- 13.33%
	Top-5: 33.33% +- 0.00%
	Top-10: 33.33% +- 0.00%


In [19]:
freqs = [len(metadata[metadata.ethnicity_category == anc].index.tolist()) for anc in rel_ancs]
accs_df_eu = pd.DataFrame({'ancestry':rel_ancs, 'n':freqs, 'top-n accuracy':all_ancs_accs_eu, 'top-n std. dev':all_ancs_stdevs_eu})
accs_df_others = pd.DataFrame({'ancestry':rel_ancs, 'n':freqs, 'top-n accuracy':all_ancs_accs_others, 'top-n std. dev':all_ancs_stdevs_others})

In [36]:
accs_df_eu
# accs_df_others

,ancestry,n,top-n accuracy,top-n std. dev
0,African,5,"[0.72, 0.9199999999999999, 0.96]","[0.09797958971132716, 0.09797958971132709, 0.0..."
1,Asian,89,"[0.651685393258427, 0.8831460674157302, 0.9146...","[0.0449438202247191, 0.011458470817062418, 0.0..."
2,European,156,"[0.6692307692307693, 0.8794871794871794, 0.928...","[0.030230323391157943, 0.022718006598294024, 0..."
3,Mixed ancestry,6,"[0.6333333333333333, 0.9666666666666668, 1.0]","[0.16329931618554522, 0.06666666666666667, 0.0]"
4,Others,6,"[0.03333333333333333, 0.19999999999999998, 0.5]","[0.06666666666666667, 0.1247219128924647, 0.14..."
5,Unknown,3,"[0.13333333333333333, 0.3333333333333333, 0.33...","[0.1632993161855452, 0.0, 0.0]"


In [25]:
## Per SUB-ancestry performance (EU+EU*)
# relevant columns: ethnicity_subgroup
all_sub_ancs_accs_eu = []
all_sub_ancs_stdevs_eu = []
rel_sub_ancs = np.unique(metadata.ethnicity_subgroup.values)
for anc in rel_sub_ancs:
    anc_mask = metadata[metadata.ethnicity_subgroup == anc].index.tolist()
    print(f"{anc} (n={len(anc_mask)}):")
    top_n_accs = []
    top_n_stdevs = []
    for n in [1,5,10]:
        accs = np.sum((correct_ranks_EU[anc_mask])<n, axis=0)/(len(correct_ranks_EU[anc_mask]))
        acc = np.mean(accs)
        stdev = np.std(accs)
        print(f"\tTop-{n}: {acc*100:.2f}% +- {stdev*100:.2f}%")
        top_n_accs.append(acc)
        top_n_stdevs.append(stdev)
    all_sub_ancs_accs_eu.append(top_n_accs)
    all_sub_ancs_stdevs_eu.append(top_n_stdevs)

African - North (n=4):
	Top-1: 85.00% +- 12.25%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
African - Sub-Saharan (n=1):
	Top-1: 20.00% +- 40.00%
	Top-5: 60.00% +- 48.99%
	Top-10: 80.00% +- 40.00%
American - Latin/Hispanic (n=4):
	Top-1: 5.00% +- 10.00%
	Top-5: 30.00% +- 18.71%
	Top-10: 40.00% +- 12.25%
Asian (n=2):
	Top-1: 50.00% +- 0.00%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
Asian - East (n=3):
	Top-1: 53.33% +- 16.33%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
Asian - South/Indian (n=11):
	Top-1: 54.55% +- 9.96%
	Top-5: 69.09% +- 4.45%
	Top-10: 78.18% +- 7.27%
Asian - West/Middle Eastern (n=73):
	Top-1: 67.67% +- 4.95%
	Top-5: 90.41% +- 1.73%
	Top-10: 92.88% +- 1.03%
European (n=156):
	Top-1: 66.92% +- 3.02%
	Top-5: 87.95% +- 2.27%
	Top-10: 92.82% +- 1.18%
Mixed ancestry (n=6):
	Top-1: 63.33% +- 16.33%
	Top-5: 96.67% +- 6.67%
	Top-10: 100.00% +- 0.00%
Oceanian/Pacific Islander (n=2):
	Top-1: 0.00% +- 0.00%
	Top-5: 0.00% +- 0.00%
	Top-10: 70.00% +- 24.49%


In [26]:
## Per SUB-ancestry performance (EU+Others)
# relevant columns: ethnicity_subgroup
all_sub_ancs_accs_others = []
all_sub_ancs_stdevs_others = []
rel_sub_ancs = np.unique(metadata.ethnicity_subgroup.values)
for anc in rel_sub_ancs:
    anc_mask = metadata[metadata.ethnicity_subgroup == anc].index.tolist()
    print(f"{anc} (n={len(anc_mask)}):")
    top_n_accs = []
    top_n_stdevs = []
    for n in [1,5,10]:
        accs = np.sum((correct_ranks_others[anc_mask])<n, axis=0)/(len(correct_ranks_others[anc_mask]))
        acc = np.mean(accs)
        stdev = np.std(accs)
        print(f"\tTop-{n}: {acc*100:.2f}% +- {stdev*100:.2f}%")
        top_n_accs.append(acc)
        top_n_stdevs.append(stdev)
    all_sub_ancs_accs_others.append(top_n_accs)
    all_sub_ancs_stdevs_others.append(top_n_stdevs)

African - North (n=4):
	Top-1: 85.00% +- 12.25%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
African - Sub-Saharan (n=1):
	Top-1: 100.00% +- 0.00%
	Top-5: 100.00% +- 0.00%
	Top-10: 100.00% +- 0.00%
American - Latin/Hispanic (n=4):
	Top-1: 5.00% +- 10.00%
	Top-5: 50.00% +- 0.00%
	Top-10: 50.00% +- 0.00%
Asian (n=2):
	Top-1: 70.00% +- 24.49%
	Top-5: 90.00% +- 20.00%
	Top-10: 100.00% +- 0.00%
Asian - East (n=3):
	Top-1: 66.67% +- 0.00%
	Top-5: 93.33% +- 13.33%
	Top-10: 100.00% +- 0.00%
Asian - South/Indian (n=11):
	Top-1: 49.09% +- 14.77%
	Top-5: 69.09% +- 10.91%
	Top-10: 76.36% +- 9.27%
Asian - West/Middle Eastern (n=73):
	Top-1: 77.81% +- 2.36%
	Top-5: 89.04% +- 1.23%
	Top-10: 93.15% +- 0.87%
European (n=156):
	Top-1: 70.13% +- 2.71%
	Top-5: 88.72% +- 1.44%
	Top-10: 92.18% +- 1.37%
Mixed ancestry (n=6):
	Top-1: 73.33% +- 17.00%
	Top-5: 96.67% +- 6.67%
	Top-10: 100.00% +- 0.00%
Oceanian/Pacific Islander (n=2):
	Top-1: 20.00% +- 24.49%
	Top-5: 80.00% +- 24.49%
	Top-10: 100.00% +- 0.

In [30]:
freqs = [len(metadata[metadata.ethnicity_subgroup == anc].index.tolist()) for anc in rel_sub_ancs]
accs_df_sub_eu = pd.DataFrame({'ancestry':rel_sub_ancs, 'n':freqs, 'top-n accuracy':all_sub_ancs_accs_eu, 'top-n std. dev':all_sub_ancs_stdevs_eu})
accs_df_sub_other = pd.DataFrame({'ancestry':rel_sub_ancs, 'n':freqs, 'top-n accuracy':all_sub_ancs_accs_others, 'top-n std. dev':all_sub_ancs_stdevs_others})

In [38]:
# accs_df_sub_eu
accs_df_sub_other

,ancestry,n,top-n accuracy,top-n std. dev
0,African - North,4,"[0.85, 1.0, 1.0]","[0.1224744871391589, 0.0, 0.0]"
1,African - Sub-Saharan,1,"[1.0, 1.0, 1.0]","[0.0, 0.0, 0.0]"
2,American - Latin/Hispanic,4,"[0.05, 0.5, 0.5]","[0.10000000000000002, 0.0, 0.0]"
3,Asian,2,"[0.7, 0.9, 1.0]","[0.2449489742783178, 0.2, 0.0]"
4,Asian - East,3,"[0.6666666666666666, 0.9333333333333332, 1.0]","[0.0, 0.13333333333333336, 0.0]"
5,Asian - South/Indian,11,"[0.4909090909090909, 0.6909090909090909, 0.763...","[0.14770978917519925, 0.10909090909090914, 0.0..."
6,Asian - West/Middle Eastern,73,"[0.7780821917808218, 0.8904109589041095, 0.931...","[0.02356801443025376, 0.01225242727397145, 0.0..."
7,European,156,"[0.7012820512820512, 0.8871794871794872, 0.921...","[0.027075271899926753, 0.01439098994913058, 0...."
8,Mixed ancestry,6,"[0.7333333333333333, 0.9666666666666666, 1.0]","[0.16996731711975951, 0.06666666666666665, 0.0]"
9,Oceanian/Pacific Islander,2,"[0.2, 0.8, 1.0]","[0.24494897427831783, 0.2449489742783178, 0.0]"


In [ ]:
## Used for Fig 5a - COHEN
# Data averaged over 5 runs (using random sampling from training splits) - GMDB v1.1.0

labels = np.array([
    "[AF] North African", "Sub-Saharan", "American - Latin/Hispanic", "Asian", "East Asian", "South Asian/Indian", "West Asian/Middle Eastern", "European", "Mixed ancestry", "Oceanian/Pacific"
])

# Top-1 accuracies of EU + non-EU, ordered by labels above
all_avg_t1 = np.array([
    85.00, 100.00, 5.00, 70.00, 66.67, 49.09, 77.81, 70.13, 73.33, 20.00
])

# Top-5 accuracies of EU + non-EU, ordered by labels above
all_avg_t5 = np.array([
    100.00, 100.00, 50.00, 90.00, 93.33, 69.09, 89.04, 88.72, 96.67, 80.00
])

# Top-1 accuracies of EU + EU*, ordered by labels above
eu_avg_t1 = np.array([
    85.00, 20.00, 5.00, 50.00, 53.33, 54.55, 67.67, 66.92, 63.33, 0.00
])

# Top-5 accuracies of EU + EU*, ordered by labels above
eu_avg_t5 = np.array([
    100.00, 60.00, 30.00, 100.00, 100.00, 69.09, 90.41, 87.95, 96.67, 0.00
])

## Sort options for labels
std_top1_eu_noneu = np.array([12.25, 0.00, 10.00, 24.50, 0.00, 14.77, 2.36, 2.71, 17.00, 24.49])
sort_order = np.argsort(std_top1_eu_noneu)
sorted_labels = labels[sort_order]
second_y = std_top1_eu_noneu[sort_order]
all_avg_t1 = all_avg_t1[sort_order]
all_avg_t5 = all_avg_t5[sort_order]
eu_avg_t1 = eu_avg_t1[sort_order]
eu_avg_t5 = eu_avg_t5[sort_order]

sort_title = "acc std top-1*"

## End: Sorting

fig, ax = plt.subplots(layout='constrained')
# ax.grid(True, which='major', axis='y', zorder=0)

x = np.arange(len(labels))
width = 0.3

offset = 0.15
rects = ax.bar(x + offset, eu_avg_t5, width/2, label="Top-5", color="skyblue", zorder=3)
rects = ax.bar(x + offset, eu_avg_t1, width, label="Top-1", color="dodgerblue", zorder=3)

offset = width + 0.15
rects = ax.bar(x + offset, all_avg_t5, width/2, label="Top-5", color="khaki",zorder=3)
# ax.errorbar(x + offset+0.1, all_avg_t5, yerr=all_std_t5, fmt="none", capsize=2, color="black")
rects = ax.bar(x + offset, all_avg_t1, width, label="Top-1", color="gold", zorder=3)
# ax.errorbar(x + offset-0.1, all_avg_t1, yerr=all_std_t1, fmt="none", capsize=2, color="black")

ax.set_title(f"Top-1 and -5 Accuracy on Cohen per Ancestral Group")
ax.set_ylabel("Accuracy (%)")
ax.set_xticks(x+width, labels, rotation=85)
# ax.legend()
ax.set_ylim(0, 100)

from matplotlib.lines import Line2D

custom_lines = [Line2D([0], [0], color="dodgerblue", lw=5, label='EU + EU*'),
                Line2D([0], [0], color="gold", lw=5, label='EU + non-EU')]

## Legend
# ax.legend(custom_lines, ['EU + EU', 'EU + nonEU'], loc='lower left')
extra_lines = [plt.Line2D([], [], linestyle='--', label='Mean top-1', color='black'),
               plt.Line2D([], [], linestyle=':', label='Mean top-5', color='black'),
               plt.Line2D([], [], linestyle='-', label='Standard deviation', color='limegreen')]
ax.legend(handles=custom_lines + extra_lines, loc='center left', bbox_to_anchor=(1.15, 0.5))

plt.axhline(y=np.mean(all_avg_t1), color='goldenrod', ls='--', zorder=2)
plt.axhline(y=np.mean(all_avg_t5), color='goldenrod', ls=':', zorder=2)
print(np.mean(all_avg_t1))
print(np.mean(all_avg_t5))

plt.axhline(y=np.mean(eu_avg_t1), color='dodgerblue', ls='--', zorder=2)
plt.axhline(y=np.mean(eu_avg_t5), color='dodgerblue', ls=':', zorder=2)
print(np.mean(eu_avg_t1))
print(np.mean(eu_avg_t5))

# draw a vertical line at location X
# plt.axvline(x = 7.8, color = 'red')

# add second y-axis
ax2 = ax.twinx()
# ax2.plot(np.append(x, x[-1]+0.6), second_y+[second_y[-1]], color='limegreen')
ax2.plot(x+0.3, second_y, color='limegreen')
ax2.set_ylabel("Average standard deviation (%)")
ax2.set_ylim(0,100)

plt.savefig('../paper_plots/fig5a_classification_accuracy_anc_COHEN.svg', dpi=300)
plt.show()